# 04 · XGBoost: önümüzdeki 5 günde hisse piyasayı geçer mi?

**XGBoost**: art arda küçük karar ağaçları kurar; her yeni ağaç öncekilerin hatasını düzeltir
(gradient boosting). Tablo verisinde en güçlü yöntemlerden biri.

**Uyarı (en önemli ders):** borsa getirisi tahmini sinyal/gürültü oranı çok düşük, kuralları zamanla değişen
bir problem. Burada gerçekçi beklenti AUC ≈ 0,50–0,55. 0,60'ın üstünü görürsen önce **hata (sızıntı)** ara.
Bu notebook'un amacı para kazandıran model değil, **doğru değerlendirme** öğrenmek:

1. **Look-ahead:** t günündeki özellik t'den sonraki veriyi kullanmamalı.
2. **Sızıntı (leakage):** hedefle örtüşen bilgi özelliklere karışmamalı.
3. **Aşırı uyum:** rastgele train/test ayrımı zaman serisinde yanlış → **walk-forward** + embargo.
4. **Baseline:** "her zaman çoğunluk sınıfı" ve lojistik regresyonu geçemiyorsan XGBoost bir şey katmıyor.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))  # learn/ klasöründen çalıştır
import kontrol as K
import papidata as pdx

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (11, 4)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [ ]:
px = pd.read_csv(pdx.DATA_DIR / "prices_adj.csv", index_col=0, parse_dates=True)
fx = pdx.usdtry()
TICKER = "THYAO"
r = np.log(px).diff()

## 1. Özellikler

İki örnek verdim. En az 5 tane daha ekle; her biri için "bu neden işe yarayabilir?" diye bir satır yorum yaz.
Fikirler: 20/60 günlük momentum, 50 günlük ortalamaya uzaklık, RSI(14), oynaklık oranı (5g/60g),
XU100'ün 5 günlük getirisi, USD/TRY'nin 5 günlük değişimi, son 5 günde hissenin piyasaya göre fazla getirisi.
**Hepsi t gününde bilinen bilgilerle** hesaplanmalı (`rolling` geriye bakar, `shift(-k)` ileriye bakar: YASAK).

In [ ]:
def build_features(px: pd.DataFrame, fx: pd.Series, ticker: str) -> pd.DataFrame:
    """index = px.index; her satır o günün kapanışında bilinen özellikler."""
    p = px[ticker]
    r = np.log(p).diff()
    f = pd.DataFrame(index=px.index)
    f["mom_5"] = np.log(p / p.shift(5))         # son 5 günün getirisi (kısa vadeli momentum/geri dönüş)
    f["vol_20"] = r.rolling(20).std()            # son 20 günün oynaklığı (risk rejimi)
    # TODO: en az 5 özellik daha
    return f

In [ ]:
K.no_lookahead(build_features, px, fx, TICKER)
X_all = build_features(px, fx, TICKER)
X_all.describe().T

## 2. Hedef

y_t = 1 eğer sonraki 5 günde (t+1 … t+5) hissenin log getirisi − XU100'ün log getirisi > 0, değilse 0.
Piyasaya göre tanımlıyoruz çünkü "borsa yükselir mi?" tahmini apayrı (ve daha zor) bir problem.

In [ ]:
def make_target(r_stock: pd.Series, r_market: pd.Series, horizon: int = 5) -> pd.Series:
    """1.0 / 0.0, son `horizon` gün NaN. r_*: günlük log getiriler (aynı index)."""
    # TODO
    raise NotImplementedError

In [ ]:
K.make_target(make_target)
y_all = make_target(r[TICKER], r["XU100"])
data = X_all.join(y_all.rename("y")).dropna()
print(data.shape, "pozitif oranı:", round(data["y"].mean(), 3))

## 3. Walk-forward doğrulama

Train: başlangıçtan test başına kadar (genişleyen pencere, en az 3 yıl). Test: sonraki 6 ay. Sonra 6 ay
kaydır. **Embargo:** hedef 5 gün ileriye baktığı için train'in son 5 günü testle örtüşür; at.

In [ ]:
def walk_forward_splits(dates: pd.DatetimeIndex, train_years: int = 3, test_months: int = 6, embargo_days: int = 5):
    """(train_pozisyonları, test_pozisyonları) çiftleri üreten generator (np.ndarray, dates içindeki konumlar).

    İlk test dates[0] + train_years'ta başlar; her test test_months sürer; train = testten önceki her şey,
    son embargo_days günü hariç. Test dönemleri çakışmaz.
    """
    # TODO
    raise NotImplementedError

In [ ]:
K.walk_forward_splits(walk_forward_splits)

## 4. Modeller

Parametrelerle oyna (`max_depth`, `n_estimators`, `learning_rate`, `subsample`, `min_child_weight`).
Derin ağaç + çok tur = aşırı uyum. Her değişiklikte sadece test ortalamasına değil, **dönemler arası
dağılıma** da bak.

In [ ]:
FEATURES = [c for c in data.columns if c != "y"]

def evaluate(data: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows, preds = [], []
    for tr, te in walk_forward_splits(data.index):
        Xtr, ytr, Xte, yte = data.iloc[tr][features], data.iloc[tr]["y"], data.iloc[te][features], data.iloc[te]["y"]
        majority = float(ytr.mean() > 0.5)
        logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
        xgb = XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            min_child_weight=20, eval_metric="logloss",
        ).fit(Xtr, ytr)
        p_log, p_xgb = logit.predict_proba(Xte)[:, 1], xgb.predict_proba(Xte)[:, 1]
        rows.append({
            "test_başı": data.index[te[0]].date(),
            "acc çoğunluk": accuracy_score(yte, np.full(len(yte), majority)),
            "AUC lojistik": roc_auc_score(yte, p_log),
            "AUC xgb": roc_auc_score(yte, p_xgb),
            "acc xgb": accuracy_score(yte, p_xgb > 0.5),
        })
        preds.append(pd.Series(p_xgb, index=data.index[te]))
    res = pd.DataFrame(rows).set_index("test_başı")
    evaluate.last_model, evaluate.last_preds = xgb, pd.concat(preds)
    return res

res = evaluate(data, FEATURES)
display(res)
res.agg(["mean", "std"])

In [ ]:
imp = pd.Series(evaluate.last_model.get_booster().get_score(importance_type="gain")).sort_values()
imp.plot.barh(title="Son modelde özellik önemi (gain)");

## 5. Sızıntı deneyi

Bilerek hata yapalım: geleceği (sonraki 5 günün getirisini) özellik olarak ekle. AUC ne oluyor?
Gerçek projelerde sızıntı bu kadar açık olmaz: ör. "bugün kapanıştan SONRA açıklanan KAP haberi"ni
bugünün özelliği yapmak, ya da bedelsiz düzeltmesini geleceği bilerek yapmak.

In [ ]:
leaky = data.copy()
leaky["GELECEK_5g"] = (r[TICKER] - r["XU100"])[::-1].rolling(5).sum()[::-1].shift(-1).reindex(leaky.index)
evaluate(leaky, FEATURES + ["GELECEK_5g"]).agg(["mean"])

## 6. Basit strateji: model para kazandırır mıydı?

Kural: XGBoost olasılığı > eşik ise o günden 5 gün THYAO'da kal, değilse TI1 benzeri nakitte (burada
basitlik için 0 getiri). Her pozisyon değişiminde komisyon öde. Karşılaştırma: hep THYAO'da kalmak.

In [ ]:
THRESH, COST = 0.55, 0.002  # %0,2 alış+satış toplam maliyet varsayımı
p = evaluate.last_preds
daily = r[TICKER].reindex(p.index)
signal = (p > THRESH).astype(float).shift(1).fillna(0)  # bugünkü sinyalle YARIN pozisyon al
trades = signal.diff().abs().fillna(0)
strat = signal * daily - trades * COST
pd.DataFrame({"strateji": strat.cumsum(), "hep THYAO": daily.cumsum()}).plot(title="Kümülatif log getiri (test dönemleri)")
print(f"işlem sayısı: {int(trades.sum())}, piyasada kalma oranı: %{signal.mean()*100:.0f}")

## 7. Yorumla

1. XGBoost lojistik regresyonu ve çoğunluk baseline'ını anlamlı biçimde geçiyor mu? Dönemler arası std'ye bak.
2. Sızıntı deneyinde AUC kaça çıktı? Gerçek bir modelde bunu nasıl fark ederdin?
3. Strateji komisyon sonrası hâlâ iyi mi? `THRESH` ve `COST`'u değiştir; sonuç ne kadar hassas?
4. **İleri:** 03'teki KAP olaylarından özellik ekle (ör. "son 5 günde P1 olay sayısı", "son olayın tipi").
   papi'nin farkı tam burada: herkesin fiyat verisi var, yapılandırılmış KAP olay verisi yok.

*Cevaplar:*

## papi'ye taşıma (sonra)

Modeli papi'ye almak = günlük tahmin üret, `forecast` tablosuna **dondurulmuş** kaydet, 5 gün sonra
`outcome` ile skorla (PRD 05 ve 11: tahminler geriye dönük değiştirilemez). Böylece modelin canlı
performansını backtest'ten ayrı izlersin; asıl öğretici olan bu farktır.